# Incremental Loading & CDC

## Objective

Identify and process only new or changed customer records from PostgreSQL.

### Learning Goals

- Incremental loading
- Watermark concept
- Change Data Capture (CDC)
- Delta MERGE foundations

### Source

- PostgreSQL customers table

### Output

- Incremental customer dataset

## Watermark Concept

A watermark represents the latest successfully processed timestamp.

Only records updated after the watermark are retrieved during incremental loads.

In [0]:
watermark = "2026-11-22 19:31:51.444"

print(watermark)

In [0]:
jdbc_url = (
    "jdbc:postgresql://"
    "ep-hidden-scene-bind3jz7-pooler.c-5.ap-southeast-1.aws.neon.tech:5432/neondb"
)

## Incremental JDBC Query

Fetch only records updated after the previous watermark.

In [0]:
incremental_query = f"""
(
SELECT *
FROM customers
WHERE last_updated_timestamp > '2026-11-21 20:30:38.240420'
) customer_updates
"""


In [0]:
incremental_df = (
    spark.read
         .format("jdbc")
         .option("url", jdbc_url)
         .option("dbtable", incremental_query)
         .option("user", "neondb_owner")
         .option("password", "npg_UnJmC6vfAi8K")
         .option("driver", "org.postgresql.Driver")
         .load()
)

In [0]:
display(incremental_df)

In [0]:
incremental_df.count()

## MERGE Incremental Changes

Apply changed customer records to the target table using Delta Lake MERGE.

### MERGE Logic

- Update existing customers
- Insert new customers


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    loyalty_points
FROM workspace.default.bronze_customers
ORDER BY customer_id

In [0]:
incremental_df.createOrReplaceTempView(
    "customer_updates"
)

In [0]:
%sql
SELECT *
FROM customer_updates

## Delta MERGE

Apply incremental customer changes to the target table.

### Matching Key

customer_id

In [0]:
%sql
MERGE INTO workspace.default.bronze_customers AS target
USING customer_updates AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.membership_type = source.membership_type,
    target.customer_status = source.customer_status,
    target.loyalty_points = source.loyalty_points,
    target.last_updated_timestamp = source.last_updated_timestamp,
    target.is_active = source.is_active
WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    city,
    membership_type,
    signup_date,
    customer_status,
    loyalty_points,
    last_updated_timestamp,
    is_active,
    source_system,
    ingestion_timestamp
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.membership_type,
    source.signup_date,
    source.customer_status,
    source.loyalty_points,
    source.last_updated_timestamp,
    source.is_active,
    'postgresql',
    current_timestamp()
)

In [0]:
%sql
SELECT
    customer_id,
    city,
    loyalty_points
FROM workspace.default.bronze_customers
WHERE customer_id = 1001

## Summary

### Concepts Covered

✅ Watermarks

✅ Incremental JDBC Loading

✅ Change Data Capture (CDC)

✅ Delta MERGE

✅ Upsert Pattern

### Key Learning

Only changed records were extracted from PostgreSQL and merged into the Delta target table without reloading the full dataset.

✅ Incremental Loading & CDC Completed